In [1]:
# ============================================
# ICU AI SYSTEM — LOS MODEL
# ============================================

# Core
import pandas as pd
import numpy as np

# Train/Test Split
from sklearn.model_selection import train_test_split

# Scaling
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# XGBoost Regression
from xgboost import XGBRegressor

# Model Saving
import joblib

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================
# LOAD LOS REGRESSION DATASET
# ============================================

df = pd.read_csv(
    "../data/processed/los_regression_processed.csv"
)

print("Dataset Loaded")

print("Shape:", df.shape)

df.head()

Dataset Loaded
Shape: (103437, 48)


,Age,Temperature,MeanArterialPressure,HeartRate,RespiratoryRate,FiO2,pO2,pCO2,ArterialpH,Sodium,...,AtmosphericPressure,SystemValue,DiagnosisValue,Gender,ApacheivScore,ApsScore,EstimatedMortalityRate,EstimatedLengthOfStay,APACHE_WARD,LOS_Hours
0,83,36.9,103.00,100.0,20.0,40.0,97.7,43.0,7.42,146.0,...,760.0,5,32,1,43,26,37.0,3.7,135,48.0
1,44,36.0,84.00,78.0,20.0,30.0,87.0,35.0,7.38,136.0,...,760.0,8,71,0,28,17,6.4,2.0,339,24.0
2,75,36.7,73.26,104.0,20.0,30.0,141.0,26.0,7.45,138.0,...,760.0,9,94,0,77,60,36.2,6.1,471,24.0
3,43,37.0,77.00,110.0,16.0,40.0,196.0,34.1,7.40,146.0,...,760.0,1,61,0,53,53,2.6,3.8,229,0.0
4,51,36.0,98.00,96.0,20.0,30.0,178.0,36.0,7.32,131.0,...,760.0,6,58,0,31,26,10.5,3.2,273,0.0


In [3]:
# ============================================
# FEATURES AND TARGET
# ============================================

X = df.drop(
    columns=["LOS_Hours"]
)

y = df["LOS_Hours"]

print("Feature Shape:", X.shape)

print("Target Shape:", y.shape)

Feature Shape: (103437, 47)
Target Shape: (103437,)


In [4]:
# ============================================
# SAVE FEATURE ORDER
# ============================================

feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/los/los_features.pkl"
)

print("LOS Feature Columns Saved")


LOS Feature Columns Saved


In [5]:
# ============================================
# TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)

print("Testing Shape:", X_test.shape)

Training Shape: (82749, 47)
Testing Shape: (20688, 47)


In [6]:
# ============================================
# FEATURE SCALING
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling Completed")

Scaling Completed


In [7]:
# ============================================
# SAVE LOS SCALER
# ============================================

joblib.dump(
    scaler,
    "../models/los/los_scaler.pkl"
)

print("LOS Scaler Saved Successfully")

LOS Scaler Saved Successfully


In [8]:
# ============================================
# TRAIN LOS REGRESSION MODEL
# ============================================

los_model = XGBRegressor(

    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    objective="reg:squarederror",

    random_state=42
)

los_model.fit(
    X_train_scaled,
    y_train
)

print("LOS Regression Model Training Completed")

LOS Regression Model Training Completed


In [9]:
# ============================================
# GENERATE PREDICTIONS
# ============================================

y_pred = los_model.predict(
    X_test_scaled
)

print("LOS Predictions Generated")

LOS Predictions Generated


In [10]:
# ============================================
# LOS MODEL EVALUATION
# ============================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:", round(mae, 2))

print("RMSE:", round(rmse, 2))

print("R² Score:", round(r2, 4))

MAE: 15.66
RMSE: 33.38
R² Score: 0.5399


In [11]:
# ============================================
# ACTUAL VS PREDICTED
# ============================================

comparison_df = pd.DataFrame({

    "Actual_LOS": y_test.values[:20],

    "Predicted_LOS": y_pred[:20]
})

comparison_df["Predicted_LOS"] = (
    comparison_df["Predicted_LOS"]
    .round(1)
)

print(comparison_df)

    Actual_LOS  Predicted_LOS
0         48.0      47.299999
1          0.0       4.200000
2          0.0       1.200000
3         24.0      36.299999
4          0.0       3.100000
5         24.0      46.400002
6         24.0      35.700001
7          0.0       0.300000
8          0.0       2.600000
9         96.0      88.199997
10        48.0      30.500000
11        48.0      45.299999
12         0.0      14.900000
13        24.0      35.400002
14        24.0       5.700000
15        48.0      52.599998
16         0.0       7.100000
17         0.0       1.000000
18        24.0      24.200001
19        96.0      54.000000


In [12]:
# ============================================
# SAVE LOS MODEL
# ============================================

joblib.dump(
    los_model,
    "../models/los/los_xgb_model.pkl"
)

print("LOS Model Saved Successfully")

LOS Model Saved Successfully
